# Surprise Potential Closed-Loop Sweep (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/waymax-simulation-experiments/blob/main/notebooks/surprise_potential_closedloop_colab.ipynb)

## Experiment Report Snapshot

### Research Question
Can surprise-potential scoring variants transfer to WOMD Waymax closed-loop simulation when the SDC planner is LatentDriver?

### Protocol
1. Bootstrap runtime with `src.platform` profile helpers.
2. Run a metric sweep with `src.workflows.surprise_potential_flow`.
3. Compare per-metric run readiness, discovery outputs, and artifact exports.

## Step 1 - Runtime Bootstrap (Repo + Drive + Deterministic Setup)

This cell is intentionally thin and delegates setup logic to `src.platform`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/achyutmorang/waymax-simulation-experiments.git'
REPO_DIR = '/content/waymax-simulation-experiments'
REPO_BRANCH = 'main'
REQUIRED_DRIVE_FOLDER = '/content/drive/MyDrive/waymax_experiments'

# Minimal pre-bootstrap so src imports work on first run.
content_root = Path('/content')
content_root.mkdir(parents=True, exist_ok=True)
try:
    _ = os.getcwd()
except FileNotFoundError:
    os.chdir(str(content_root))

if not Path(REPO_DIR).exists():
    subprocess.run(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)

for p in (REPO_DIR, f"{REPO_DIR}/src"):
    if p not in sys.path:
        sys.path.insert(0, p)

from src.platform import (
    bootstrap_colab_runtime_with_config,
    surprise_potential_colab_runtime_config,
)

runtime_cfg = surprise_potential_colab_runtime_config(
    repo_url=REPO_URL,
    repo_branch=REPO_BRANCH,
    repo_dir=REPO_DIR,
    required_drive_folder=REQUIRED_DRIVE_FOLDER,
)
runtime_result = bootstrap_colab_runtime_with_config(runtime_cfg)
print('[bootstrap] repo_dir =', runtime_result.prepared_repo_dir)
print('[bootstrap] repo_rev =', runtime_result.repo_sync.repo_rev)
print('[bootstrap] drive_ready =', runtime_result.drive_status.writable)

## Step 2 - Workflow API Imports

In [ ]:
from src.workflows import (
    report_surprise_potential_sweep,
    run_surprise_potential_metric_sweep,
)

## Step 3 - User Configuration

Set a compact metric list for pilot runs first, then expand once runtime behavior is stable.

In [ ]:
# Surprise metrics available in this pipeline:
# latent_belief_kl | predictive_kl | predictive_w2 | predictive_seq_kl | predictive_seq_w2 | action_kl
METRICS = [
    'latent_belief_kl',
    'predictive_seq_w2',
]

# Paper-style counterfactual family labels:
# fut_none | fut_gt | hist_rmv | fut_cvm | fut_cvm_l | fut_pred | hist_prim | fut_prim
COUNTERFACTUAL_FAMILIES = [
    'hist_prim',
]

PERSIST_ROOT = '/content/drive/MyDrive/waymax_experiments/closedloop_runs/v1'
RUN_TAG_BASE = ''  # Leave empty to auto-create: <RUN_TAG_PREFIX>_<UTCSTAMP>_<family>_<metric>
RUN_TAG_PREFIX = 'surprise_potential'
RUN_MODE = 'fresh'  # fresh | resume | auto

N_SHARDS = 5
SHARD_ID = 'auto'
PLANNER_BACKEND = 'latentdriver'
CONTINUE_ON_ERROR = True

QUICK_PROBE_SETTINGS = {
    'quick_probe_scenarios': 1,
    'quick_probe_proposals_per_scenario': 4,
    'stop_if_quick_probe_collapsed': False,
    'auto_escalate_quick_probe': True,
    'max_probe_escalations': 3,
    'probe_scale_multipliers': (1.0, 1.35, 1.8),
    'probe_delta_l2_multipliers': (1.0, 1.2, 1.4),
    'probe_delta_clip_multipliers': (1.0, 1.1, 1.2),
    'probe_budget_bump_per_escalation': 2,
    'apply_successful_probe_tuning': True,
}

SWEEP_KWARGS = {
    'n_eval_scenarios': 100,
    'strict_min_eval': 100,
    'n_total_scenarios_floor': 400,
    'quick_probe_enabled': True,
    'restore_from_upload': False,
    'stop_on_preflight_fail': False,
    'surprise_gate_enabled': True,
    'stop_on_gate_fail': False,
    'allow_main_loop_when_gate_fails': False,
    'warn_on_config_drift': True,
}


## Step 4 - Run Metric Sweep

All heavy orchestration is delegated to `src.workflows.surprise_potential_flow`.

In [ ]:
sweep_bundle = run_surprise_potential_metric_sweep(
    metrics=METRICS,
    counterfactual_families=COUNTERFACTUAL_FAMILIES,
    persist_root=PERSIST_ROOT,
    run_tag_base=RUN_TAG_BASE,
    run_tag_prefix=RUN_TAG_PREFIX,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    run_mode=RUN_MODE,
    planner_backend=PLANNER_BACKEND,
    continue_on_error=bool(CONTINUE_ON_ERROR),
    quick_probe_settings=QUICK_PROBE_SETTINGS,
    **SWEEP_KWARGS,
)

report_surprise_potential_sweep(sweep_bundle, display_fn=display, preview_rows=40)


## Step 5 - Reporting Tables

In [ ]:
summary_df = sweep_bundle.summary_df
method_rollup_df = sweep_bundle.method_rollup_df
artifacts_df = sweep_bundle.artifacts_df
errors_df = sweep_bundle.errors_df

display(summary_df)
display(method_rollup_df)
display(errors_df)

display(artifacts_df.head(50))

for key, run_bundle in sweep_bundle.metric_runs.items():
    print(
        f"\n[run] {key} family={run_bundle.counterfactual_family} "
        f"metric={run_bundle.metric} run_tag={run_bundle.run_tag}"
    )
    if not run_bundle.signal_bundle.signal_summary_df.empty:
        display(run_bundle.signal_bundle.signal_summary_df)


## Notes
- This notebook is intentionally orchestration-only.
- Add new experiment logic in `src/workflows/surprise_potential_flow.py`, not in notebook cells.